# Reaction-Diffusion Simulator

This notebook runs the `ReactionDiffusion` simulator — a coupled PDE system modelling two interacting species $u$ and $v$ with diffusion and nonlinear reaction kinetics.

- **State variables**: `u` and `v` (species concentrations).
- **Conditioning variables**: `beta` (reaction rate strength), `d` (ratio of diffusion coefficients).
- **Dynamics**: spectral (FFT-based) spatial differentiation combined with `scipy` RK45 time integration.
- **Boundary conditions**: periodic in both spatial directions.

### Why this is useful

Reaction-diffusion systems produce a rich variety of spatiotemporal patterns — stripes, spots, spirals, and labyrinthine textures — depending on the parameter regime. This makes them a natural benchmark for learning PDEs that exhibit diverse structure from low-dimensional parameters.


## Governing equations

The two-species reaction-diffusion system takes the form:

$$
\frac{\partial u}{\partial t} = D_u \nabla^2 u + R_u(u, v; \beta),
\qquad
\frac{\partial v}{\partial t} = D_v \nabla^2 v + R_v(u, v; \beta),
$$

where $D_v / D_u = d$ and $R_u$, $R_v$ are the nonlinear reaction terms parameterised by $\beta$.

### Boundary conditions

Periodic in both spatial directions on the domain $[-L/2, L/2]^2$:

$$
u(-L/2, y, t) = u(L/2, y, t), \qquad u(x, -L/2, t) = u(x, L/2, t),
$$

and likewise for $v$.

### Parameters

- **`beta`**: controls the strength of the reaction nonlinearity. Larger values push the system further from equilibrium.
- **`d`**: ratio of diffusion coefficients $D_v / D_u$. Small `d` means species $v$ diffuses much more slowly, favouring pattern formation.


In [ ]:
from IPython.display import HTML

from autosim.experimental.simulations import ReactionDiffusion
from autosim.utils import plot_spatiotemporal_video

In [ ]:
sim = ReactionDiffusion(
    return_timeseries=True,
    log_level="progress_bar",
    n=64,
    L=20,
    T=32.2,
    dt=0.1,
    parameters_range={
        "beta": (1.0, 2.0),
        "d": (0.05, 0.3),
    },
)

batch = sim.forward_samples_spatiotemporal(n=2, random_seed=42)

print("data shape:", batch["data"].shape, "[batch, time, x, y, channels={u, v}]")
print("constant_scalars shape:", batch["constant_scalars"].shape)
print("sampled params (beta, d):", batch["constant_scalars"])

In [ ]:
anim = plot_spatiotemporal_video(
    batch["data"],
    batch_idx=0,
    channel_names=["u", "v"],
    preserve_aspect=True,
)
HTML(anim.to_jshtml())

## Dataset grid comparison

Load saved datasets from disk and display a snapshot from each in a tiled grid.
The bottom row spans the full width at half height for the 4:1 aspect-ratio dataset.

In [ ]:
import torch
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from pathlib import Path

# --- dataset list (imagine all 7 exist on disk; missing ones get a placeholder) ---
datasets = [
    "gpe_laser_and_speckle",
    "gpe_laser_only",
    "gpe_speckle_only",
    "gpe_vortex_lattice",
    "gpe_low_complexity",
    "gpe_high_complexity",
    "gpe_obstacle_motion",  # 4:1 aspect ratio — wide row
]

base_dir = Path("generated_datasets")
batch_idx, time_idx, channel = 0, -1, 0  # which snapshot to show

# --- build grid: 3 rows × 2 cols + 1 wide row (half height) ---
fig = plt.figure(figsize=(10, 12))
gs = GridSpec(4, 2, height_ratios=[1, 1, 1, 0.5], hspace=0.35, wspace=0.2)

# first 6 datasets in 3×2
for i, name in enumerate(datasets[:6]):
    row, col = divmod(i, 2)
    ax = fig.add_subplot(gs[row, col])
    path = base_dir / name / "train" / "data.pt"
    if path.exists():
        data = torch.load(path, weights_only=False)["data"]
        ax.imshow(data[batch_idx, time_idx, :, :, channel].numpy(), cmap="viridis")
    else:
        ax.text(0.5, 0.5, "not found", transform=ax.transAxes, ha="center", va="center", color="grey")
    ax.set_title(name.replace("gpe_", "").replace("_", " ").title(), fontsize=10)
    ax.axis("off")

# 7th dataset — spans full width, half height (4:1)
ax_wide = fig.add_subplot(gs[3, :])
name = datasets[6]
path = base_dir / name / "train" / "data.pt"
if path.exists():
    data = torch.load(path, weights_only=False)["data"]
    ax_wide.imshow(data[batch_idx, time_idx, :, :, channel].numpy(), cmap="viridis", aspect="auto")
else:
    ax_wide.text(0.5, 0.5, "not found", transform=ax_wide.transAxes, ha="center", va="center", color="grey")
ax_wide.set_title(name.replace("gpe_", "").replace("_", " ").title(), fontsize=10)
ax_wide.axis("off")

plt.suptitle("GPE Dataset Comparison", fontsize=14)
plt.tight_layout()
plt.show()